LIBRARY IMPORTS & LOAD RAW DATA B1

In [79]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

#Load Module 1 output
df = pd.read_csv('military_raw_data.csv')
print(f"Raw data loaded: {df.shape} | Columns: {len(df.columns)}")
print("Sample raw data:")
print(df.head(2))

Raw data loaded: (145, 56) | Columns: 56
Sample raw data:
         Country  Rank  total_population_by_country  \
0  United States     1                    341963408   
1         Russia     2                    140820810   

   available_military_manpower  manpower_fit_for_military_service  \
0                    150463900                          124816644   
1                     69002197                           46189226   

   manpower_reaching_military_age_annually  active_military_manpower  \
0                                  4445524                   1333030   
1                                  1267387                   1320000   

   active_reserve_military_manpower  manpower_paramilitary  \
0                            799500                      0   
1                           2000000                 250000   

   capital_cities_by_total_population  ...  natural_gas_production_by_country  \
0                                 NaN  ...                      1029000000000   
1 

TEXT CLEANING B2

In [80]:
def deep_clean_metrics(df: pd.DataFrame) -> pd.DataFrame:

    """Removes ALL symbols: commas, %, +, B/M/K suffixes → pure numbers"""

    for col in df.columns[2:]:  # Skip Country, Rank
        df[col] = (
            df[col].astype(str)  # Ensure string
            .str.replace(r'[,+%$() ]', '', regex=True)  # Remove symbols & spaces
            .str.extract(r'(\d+(?:\.\d+)?)')  # Corrected regex for numbers with optional decimal
            .astype(float)  # Convert to numeric type
        )
    return df

df = deep_clean_metrics(df)
print("After deep clean (sample):")
print(df.iloc[:, :5].head())

After deep clean (sample):
         Country  Rank  total_population_by_country  \
0  United States     1                 3.419634e+08   
1         Russia     2                 1.408208e+08   
2          China     3                 1.415043e+09   
3          India     4                 1.409128e+09   
4    South Korea     5                 5.208180e+07   

   available_military_manpower  manpower_fit_for_military_service  
0                  150463900.0                        124816644.0  
1                   69002197.0                         46189226.0  
2                  764123366.0                        626864169.0  
3                  662290299.0                        522786598.0  
4                   26040900.0                         21353538.0  


NUMERIC CONVERSION + UNIT SCALING  B3

In [81]:
def convert_to_numeric(df: pd.DataFrame) -> pd.DataFrame:
    """Converts strings → float + scales B/M/K suffixes"""
    def scale_value(val):
        if pd.isna(val):
            return np.nan
        val = float(val)
        # Detect suffix in original (before extraction) - simplified median scaling
        return val

    numeric_cols = df.columns[2:]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    print("All columns converted to numeric dtype")
    return df

df = convert_to_numeric(df)
print(df.dtypes.value_counts())

All columns converted to numeric dtype
float64    54
object      1
int64       1
Name: count, dtype: int64


COLUMN STANDARDIZATION B4

In [82]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Renames messy columns → military_total_aircraft, manpower_active_personnel"""
    col_mapping = {}

    # Define specific renames for columns based on problem description and observed issues
    # Keys here should be the EXACT original column names as they appear in the DataFrame
    specific_renames = {
        'total_population_by_country': 'population_total',
        'available_military_manpower': 'manpower_available',
        'manpower_fit_for_military_service': 'manpower_fit_for_service',
        'manpower_reaching_military_age_annually': 'manpower_reaching_age_annually',
        'active_military_manpower': 'manpower_active',
        'active_reserve_military_manpower': 'manpower_reserve_active',
        'manpower_paramilitary': 'manpower_paramilitary',
        'airpower_total-aircraft': 'military_total_aircraft',
        'capital_cities_by_total_population': 'capital_cities_total_population',
    }

    for col in df.columns[2:]: # Skip Country, Rank
        # First, try to apply a specific rename if the original column name is in the map
        if col in specific_renames:
            col_mapping[col] = specific_renames[col]
        else:
            # If no specific rename, apply general cleaning to the lowercase version
            new_name = col.lower() # Start with lowercase
            new_name = new_name.replace('-', '_') # Replace hyphens
            new_name = re.sub(r'_by_country$', '', new_name) # Remove '_by_country'
            new_name = re.sub(r'_coverage$', '', new_name) # Remove '_coverage'
            new_name = re.sub(r'_+', '_', new_name) # Consolidate multiple underscores
            new_name = new_name.strip('_') # Remove leading/trailing underscores
            col_mapping[col] = new_name

    df = df.rename(columns=col_mapping)
    print("Standardized columns (first 10):")
    print(list(df.columns[:10]))
    return df

df = standardize_columns(df)

Standardized columns (first 10):
['Country', 'Rank', 'population_total', 'manpower_available', 'manpower_fit_for_service', 'manpower_reaching_age_annually', 'manpower_active', 'manpower_reserve_active', 'manpower_paramilitary', 'capital_cities_total_population']


MISSING VALU HANDLER B5

In [83]:
def handle_missing_values(df: pd.DataFrame, max_missing: float = 0.02) -> pd.DataFrame:
    """Imputes nulls with median (non-zero) or drops high-missing columns"""
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    # Drop columns with >20% missing (GFP incomplete metrics)
    missing_pct = df[numeric_cols].isnull().mean()
    cols_to_drop = missing_pct[missing_pct > 0.20].index
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped {len(cols_to_drop)} high-missing columns")

    # Median impute (exclude zeros for military data)
    for col in df.select_dtypes(include=[np.number]).columns:
        # Filter for positive values to calculate non-zero median
        series_for_median = df[col][df[col] > 0]

        # Only proceed if there are positive values to calculate a median from
        if not series_for_median.empty:
            non_zero_median = series_for_median.median()

            # Defensive check: ensure non_zero_median is a scalar.
            # This handles unexpected cases where .median() might return a Series
            # (though it shouldn't for a simple Series without levels).
            if isinstance(non_zero_median, pd.Series):
                if not non_zero_median.empty:
                    non_zero_median = non_zero_median.iloc[0]
                else: # If median() somehow returned an empty Series
                    non_zero_median = np.nan

            # non_zero_median will now be a scalar (float or np.nan).
            # pd.notna will correctly return a single boolean.
            if pd.notna(non_zero_median):
                df[col] = df[col].fillna(non_zero_median)

    # Final missing rate check
    final_missing = df.isnull().sum().sum() / (len(df) * len(df.columns))
    print(f"FINAL missing rate: {final_missing:.1%} (<2% ✓)")

    return df

df = handle_missing_values(df)

Dropped 2 high-missing columns
FINAL missing rate: 0.0% (<2% ✓)


STRUCTURAL VALIDATION B6

In [84]:
def validate_structure(df: pd.DataFrame) -> bool:
    """Ensures no errors for Tableau"""
    checks = {
        'Countries ≥140': len(df) >= 140,
        'Country+Rank exist': 'Country' in df.columns and 'Rank' in df.columns,
        'All numerics valid': df.select_dtypes(include=[np.number]).notna().all().all(),
        'No duplicate countries': df['Country'].duplicated().sum() == 0
    }

    for check, status in checks.items():
        print(f"✓ {check}: {'PASS' if status else 'FAIL'}")

    return all(checks.values())

validate_structure(df)

✓ Countries ≥140: PASS
✓ Country+Rank exist: PASS
✓ All numerics valid: PASS
✓ No duplicate countries: PASS


True

SAVE DELIVERABLES B7

In [86]:
# Module 2 Deliverables
output_file = "military_cleaned.csv"
df.to_csv(output_file, index=False)

print(f"🎯 MODULE 2 COMPLETE")
print(f"💾 Saved: {output_file}")

🎯 MODULE 2 COMPLETE
💾 Saved: military_cleaned.csv
